In [1]:
# set up torch
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class CNN(torch.nn.Module):

    def __init__(self, input_shape = (200,200), output_size = 10):
        super(CNN, self).__init__()

        self.conv1 = torch.nn.Conv2d(in_channels=3,kernel_size=3, out_channels=32) # 3x200x200x3 -> 32x198x198 because no padding specified
        self.pool1 = torch.nn.MaxPool2d(kernel_size=(2,2)) # 32x198x198 -> 32x99x99
        self.lin1 = torch.nn.Linear(in_features=32*99*99, out_features=64)
        self.lin2 = torch.nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        x = torch.nn.functional.relu(self.conv1(x)) # -> 32x198x198
        x = self.pool1(x) # -> 32x99x99
        x = x.flatten(start_dim=1) # -> 32*99*99 = 313632
        x = torch.nn.functional.relu(self.lin1(x))
        x = self.lin2(x)
        # x = torch.sigmoid(x)
        return x



In [3]:
model = CNN()
model.to(torch.device(device))
print(model)

CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (lin1): Linear(in_features=313632, out_features=64, bias=True)
  (lin2): Linear(in_features=64, out_features=1, bias=True)
)


In [4]:
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------


In [5]:
from torchvision import transforms
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

In [6]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_dataset = datasets.ImageFolder(root='data/train', transform=train_transforms)
validation_dataset = datasets.ImageFolder(root='data/test', transform=train_transforms)

batch_size = 20
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

In [7]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
#criterion = torch.nn.BCELoss()
criterion = torch.nn.BCEWithLogitsLoss()
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6665, Acc: 0.6112, Val Loss: 0.6511, Val Acc: 0.6617
Epoch 2/10, Loss: 0.5702, Acc: 0.6787, Val Loss: 0.6332, Val Acc: 0.6318
Epoch 3/10, Loss: 0.5207, Acc: 0.7350, Val Loss: 0.6143, Val Acc: 0.6766
Epoch 4/10, Loss: 0.4773, Acc: 0.7600, Val Loss: 0.6049, Val Acc: 0.6617
Epoch 5/10, Loss: 0.4606, Acc: 0.7550, Val Loss: 0.7307, Val Acc: 0.5672
Epoch 6/10, Loss: 0.3954, Acc: 0.8275, Val Loss: 0.6412, Val Acc: 0.6866
Epoch 7/10, Loss: 0.2844, Acc: 0.8838, Val Loss: 0.8307, Val Acc: 0.6816
Epoch 8/10, Loss: 0.2885, Acc: 0.8788, Val Loss: 0.7052, Val Acc: 0.7114
Epoch 9/10, Loss: 0.1882, Acc: 0.9313, Val Loss: 0.9275, Val Acc: 0.6866
Epoch 10/10, Loss: 0.2585, Acc: 0.8912, Val Loss: 0.8158, Val Acc: 0.6915


In [8]:
median_train_acc = np.median(history['acc'])
print(f"Median Training Accuracy: {median_train_acc:.4f}")

Median Training Accuracy: 0.7937


In [9]:
std_train_loss = np.std(history['loss'])
print(f"Standard deviation of training loss: {std_train_loss:.4f}")

Standard deviation of training loss: 0.1462


In [10]:
train_transforms_aug = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
aug_train_dataset = datasets.ImageFolder(root='data/train', transform=train_transforms_aug)
aug_validation_dataset = datasets.ImageFolder(root='data/test', transform=train_transforms_aug)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=batch_size, shuffle=True)
aug_validation_loader = DataLoader(aug_validation_dataset, batch_size=batch_size, shuffle=False)

history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in aug_train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(aug_train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in aug_validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(aug_validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")


Epoch 1/10, Loss: 0.6472, Acc: 0.6538, Val Loss: 0.6823, Val Acc: 0.6169
Epoch 2/10, Loss: 0.6121, Acc: 0.6288, Val Loss: 0.6177, Val Acc: 0.6418
Epoch 3/10, Loss: 0.5995, Acc: 0.6725, Val Loss: 0.5744, Val Acc: 0.6866
Epoch 4/10, Loss: 0.5910, Acc: 0.6713, Val Loss: 0.6057, Val Acc: 0.6965
Epoch 5/10, Loss: 0.5675, Acc: 0.6750, Val Loss: 0.5960, Val Acc: 0.6965
Epoch 6/10, Loss: 0.5496, Acc: 0.7175, Val Loss: 0.5617, Val Acc: 0.7114
Epoch 7/10, Loss: 0.5328, Acc: 0.7338, Val Loss: 0.5915, Val Acc: 0.6965
Epoch 8/10, Loss: 0.5482, Acc: 0.7075, Val Loss: 0.5970, Val Acc: 0.7015
Epoch 9/10, Loss: 0.5230, Acc: 0.7238, Val Loss: 0.5696, Val Acc: 0.7363
Epoch 10/10, Loss: 0.5338, Acc: 0.7225, Val Loss: 0.5534, Val Acc: 0.7363


In [12]:
augmented_val_losses = history['val_loss'] 
mean_val_loss = np.mean(augmented_val_losses)
print(f"Mean Validation Loss (with augmentation): {mean_val_loss:.4f}")

Mean Validation Loss (with augmentation): 0.5949


In [16]:
last_5_val_acc = history['val_acc'][-5:]  # Epochs 16-20
avg_val_acc = np.mean(last_5_val_acc)
print(f"Average Validation Accuracy over last 5 epochs (with augmentation): {avg_val_acc:.4f}")

Average Validation Accuracy over last 5 epochs (with augmentation): 0.7164
